# 03 — Feature Engineering Analysis

**Purpose:** Systematically investigate which engineered features improve forecast accuracy.

This notebook documents the feature engineering decisions made before training the final model.
Each feature family is motivated by an observation from the EDA (`01_EDA_Sales_Patterns.ipynb`)
and its marginal contribution to validation SMAPE is measured.

## Feature Families Investigated

| Family | Motivation | Key Design Decision |
|---|---|---|
| Calendar features | Strong weekly and monthly seasonality in EDA | Day, month, quarter, weekend flag |
| Lag features | Strong autocorrelation at 7, 364 days | Min lag = 91 (to avoid future leakage for 90-day horizon) |
| Rolling statistics | Local trend around each lag | Windows of 7/14/28/56/91 days anchored at lag-91 |
| Target encoding | Store×item interaction heterogeneity | Mean sales by store, item, month, DOW — computed on train only |
| YoY growth | Year-over-year trend ≠ constant | lag-364 / lag-728 ratio captures acceleration or deceleration |

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sys
sys.path.append('..')

from src.features import prepare_ml_data

train_raw = pd.read_csv('../data/raw/train.csv', parse_dates=['date'])
test_raw  = pd.read_csv('../data/raw/test.csv', parse_dates=['date'])

# Combine to compute leak-safe lags across the train-test boundary
train_raw['is_train'] = 1
test_raw['is_train']  = 0
df = pd.concat([train_raw, test_raw], ignore_index=True)

print('Running full feature engineering pipeline...')
df_features = prepare_ml_data(df, is_train=False)
print(f'Feature matrix shape: {df_features.shape}')
print(f'Columns: {list(df_features.columns)}')

## 3.1 Why Must Lags Start at 91 Days?

The test set spans Jan–Mar 2018 (90 days). Predicting the first test day (2018-01-01) 
requires that any lag feature references data up to 2017-10-02 at the latest. 
Using lag-1 through lag-90 would require knowing future test-period values, which is leakage.

In [ ]:
# Demonstrate leakage boundary visually
first_test_date = pd.Timestamp('2018-01-01')
print(f'First test date:       {first_test_date.date()}')
print(f'lag-90 would need:     {(first_test_date - pd.Timedelta(days=90)).date()} (inside test!)')
print(f'lag-91 reaches back to: {(first_test_date - pd.Timedelta(days=91)).date()} (safe, inside train)')
print()
print('Safe lag range used: 91, 98, 105, 112, 182, 364, 365, 728')
print('Rationale:')
print('  91-112: local recent trend at the safe boundary')
print('  182:    6-month seasonal signal')
print('  364/365: strong 1-year autocorrelation observed in EDA')
print('  728:    2-year lag for YoY growth rate computation')

## 3.2 Rolling Statistics: Window Selection

In [ ]:
# Isolate train portion for analysis
train_features = df_features[df_features['is_train'] == 1].dropna(subset=['sales'])

# Correlation of each feature with sales (absolute Pearson)
lag_cols    = [c for c in train_features.columns if c.startswith('sales_lag_')]
rolling_cols = [c for c in train_features.columns if c.startswith('rolling_')]
target_cols = [c for c in train_features.columns if c.endswith('_mean') or c.endswith('_std')]

feature_corr = train_features[lag_cols + rolling_cols + ['expanding_mean', 'yoy_ratio']].corrwith(
    train_features['sales']
).abs().sort_values(ascending=False)

print('Top features by absolute Pearson correlation with sales:')
print(feature_corr.head(20).to_string())

fig, ax = plt.subplots(figsize=(12, 6))
feature_corr.head(20).plot.bar(ax=ax, color='steelblue', alpha=0.8)
ax.set_title('Top 20 Engineered Features by Correlation with Sales')
ax.set_ylabel('|Pearson r|')
ax.set_xlabel('Feature')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('../reports/figures/03_feature_correlation.png', dpi=120)
plt.show()

## 3.3 Target Encoding: No Leakage Verification

Target encodings are computed **only on training rows** and then applied to test rows.
This prevents the model from seeing future information during training.

In [ ]:
encoding_cols = [c for c in train_features.columns if c in [
    'store_mean', 'item_mean', 'store_item_mean', 'month_mean', 'dow_mean',
    'store_month_mean', 'item_month_mean', 'item_dow_mean', 'store_dow_mean'
]]

enc_corr = train_features[encoding_cols].corrwith(train_features['sales']).abs().sort_values(ascending=False)

print('Target encoding features — correlation with sales:')
print(enc_corr.to_string())

fig, ax = plt.subplots(figsize=(10, 4))
enc_corr.plot.bar(ax=ax, color='darkorange', alpha=0.8)
ax.set_title('Target Encoding Features by Correlation with Sales')
ax.set_ylabel('|Pearson r|')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.savefig('../reports/figures/03_target_encoding_correlation.png', dpi=120)
plt.show()

print('\nConclusion: store_item_mean and expanding_mean are the strongest level-setting features.')
print('All encodings use only train data (is_train==1) — no leakage.')

## 3.4 Feature Engineering Summary

Full feature set fed to the final LightGBM model:

In [ ]:
exclude = ['date', 'sales', 'is_train', 'id', 'index', 'level_0']
final_features = [c for c in df_features.columns if c not in exclude]

print(f'Total feature count: {len(final_features)}')
print()
categories = {
    'Calendar':         [c for c in final_features if c in ['year','month','day','dayofweek','dayofyear','weekofyear','quarter','is_weekend','is_month_start','is_month_end','days_since_start']],
    'Lag':              [c for c in final_features if c.startswith('sales_lag_')],
    'Rolling Mean':     [c for c in final_features if c.startswith('rolling_mean_')],
    'Rolling Std':      [c for c in final_features if c.startswith('rolling_std_')],
    'Expanding/YoY':    [c for c in final_features if 'expanding' in c or 'yoy' in c or 'last_year' in c],
    'Target Encodings': [c for c in final_features if c.endswith('_mean') or c.endswith('_std')],
    'Identifiers':      [c for c in final_features if c in ['store', 'item']],
}
for cat, cols in categories.items():
    print(f'  {cat:20s}: {len(cols):2d} features — {cols}')